# B09a: Transfer Learning Fundamentals - COMPSCI 714

**Course:** COMPSCI 714 - AI Architecture and Design  
**Lecture:** 7-2 (Transfer Learning II)  

This notebook covers the fundamentals of transfer learning as presented in COMPSCI 714 Lecture 7-2.
All implementations use PyTorch and are designed to run in Google Colab.

**Key Idea:** Transfer learning leverages knowledge from a *source domain* to improve
learning in a *target domain*, especially when target data is limited.

## Table of Contents

1. [Overview of Transfer Learning](#1-overview-of-transfer-learning)
2. [Model Fine-tuning](#2-model-fine-tuning)
3. [Conservation Learning](#3-conservation-learning)
4. [Layer Transfer](#4-layer-transfer)
5. [Multitask Learning](#5-multitask-learning)
6. [Domain-Adversarial Training](#6-domain-adversarial-training)
7. [Zero-shot and Few-shot Learning](#7-zero-shot-and-few-shot-learning)
8. [Practical Example: Fine-tuning a Pretrained Model](#8-practical-example)
9. [Summary](#9-summary)

---
## Setup

Install and import required libraries.

In [ ]:
# Setup - run this cell first
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('Setup complete!')

---
## 1. Overview of Transfer Learning

### Definition

Transfer learning is a machine learning technique where knowledge gained from solving one problem
(the **source task**) is applied to a different but related problem (the **target task**).

Formally:
- **Source domain** $\mathcal{D}_S$ with task $\mathcal{T}_S$
- **Target domain** $\mathcal{D}_T$ with task $\mathcal{T}_T$
- **Goal:** Improve the target predictive function $f_T(\cdot)$ using knowledge from $\mathcal{D}_S$ and $\mathcal{T}_S$

### When to Use Transfer Learning

Transfer learning is applicable when:
1. **Domains differ:** $\mathcal{D}_S \neq \mathcal{D}_T$ (different feature spaces or distributions)
2. **Tasks differ:** $\mathcal{T}_S \neq \mathcal{T}_T$ (different label spaces or objectives)
3. **Limited target data:** Not enough labelled data in the target domain

### Examples
| Source | Target | What Transfers |
|--------|--------|----------------|
| ImageNet (1000 classes) | Medical imaging (2 classes) | Visual features |
| English speech recognition | Maori speech recognition | Acoustic features |
| Sentiment analysis (movies) | Sentiment analysis (products) | Language understanding |

In [ ]:
# Visualize the transfer learning concept
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Source domain
np.random.seed(42)
X_source = np.random.randn(200, 2) * 1.5
y_source = (X_source[:, 0] + X_source[:, 1] > 0).astype(int)
axes[0].scatter(X_source[y_source==0, 0], X_source[y_source==0, 1], c='blue', alpha=0.5, label='Class 0')
axes[0].scatter(X_source[y_source==1, 0], X_source[y_source==1, 1], c='red', alpha=0.5, label='Class 1')
axes[0].set_title('Source Domain (abundant data)', fontsize=12)
axes[0].legend()
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# Target domain (limited data, shifted)
X_target = np.random.randn(20, 2) * 1.2 + np.array([1.0, 0.5])
y_target = (X_target[:, 0] + X_target[:, 1] > 1.5).astype(int)
axes[1].scatter(X_target[y_target==0, 0], X_target[y_target==0, 1], c='blue', alpha=0.7, label='Class 0', s=80)
axes[1].scatter(X_target[y_target==1, 0], X_target[y_target==1, 1], c='red', alpha=0.7, label='Class 1', s=80)
axes[1].set_title('Target Domain (limited data)', fontsize=12)
axes[1].legend()
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

# Transfer learning flow diagram
axes[2].axis('off')
axes[2].text(0.5, 0.9, 'Transfer Learning', ha='center', fontsize=14, fontweight='bold')
axes[2].annotate('', xy=(0.5, 0.55), xytext=(0.2, 0.75),
                 arrowprops=dict(arrowstyle='->', lw=2, color='green'))
axes[2].annotate('', xy=(0.5, 0.55), xytext=(0.8, 0.75),
                 arrowprops=dict(arrowstyle='->', lw=2, color='green'))
axes[2].text(0.2, 0.78, 'Source\nKnowledge', ha='center', fontsize=10,
             bbox=dict(boxstyle='round', facecolor='lightblue'))
axes[2].text(0.8, 0.78, 'Target\nData', ha='center', fontsize=10,
             bbox=dict(boxstyle='round', facecolor='lightyellow'))
axes[2].text(0.5, 0.45, 'Improved\nTarget Model', ha='center', fontsize=11,
             bbox=dict(boxstyle='round', facecolor='lightgreen'))

plt.tight_layout()
plt.show()
print('Transfer learning uses source domain knowledge to improve target domain performance.')

---
## 2. Model Fine-tuning

### Concept

Fine-tuning is the most common transfer learning approach:
1. **Pre-train** a model on the source domain (large dataset)
2. **Fine-tune** the model on the target domain (small dataset)

### Process
```
Source Data (large) → Train Model → Pretrained Model
                                         ↓
Target Data (small) → Fine-tune → Adapted Model
```

### Challenges
- **Overfitting:** With limited target data, the model can memorize rather than generalize
- **Catastrophic forgetting:** The model may lose useful source knowledge
- **Learning rate selection:** Too high → destroy pretrained features; too low → slow adaptation

### Speaker Adaptation Example (Speech Recognition)
- Source: General speech recognition model trained on many speakers
- Target: Adapt to a specific speaker with only a few minutes of speech
- The model fine-tunes acoustic features to match the new speaker's voice characteristics

In [ ]:
# Demonstrate fine-tuning: Train on source, then fine-tune on target

# Simple neural network
class SimpleNet(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=32, output_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

# Generate source data (abundant)
torch.manual_seed(42)
X_source = torch.randn(1000, 10)
y_source = (X_source[:, 0] + X_source[:, 1] > 0).long()

# Generate target data (limited, slightly different distribution)
X_target = torch.randn(50, 10) * 0.8 + 0.2
y_target = (X_target[:, 0] + X_target[:, 1] + X_target[:, 2] > 0.5).long()

# Step 1: Pre-train on source data
model = SimpleNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

source_loader = DataLoader(TensorDataset(X_source, y_source), batch_size=64, shuffle=True)

print('Step 1: Pre-training on source domain...')
for epoch in range(20):
    for X_batch, y_batch in source_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

# Evaluate on target BEFORE fine-tuning
model.eval()
with torch.no_grad():
    preds_before = model(X_target.to(device)).argmax(dim=1)
    acc_before = (preds_before == y_target.to(device)).float().mean().item()
print(f'Target accuracy BEFORE fine-tuning: {acc_before:.3f}')

# Step 2: Fine-tune on target data (lower learning rate)
print('\nStep 2: Fine-tuning on target domain...')
optimizer_ft = optim.Adam(model.parameters(), lr=0.001)  # Lower LR!
target_loader = DataLoader(TensorDataset(X_target, y_target), batch_size=16, shuffle=True)

model.train()
for epoch in range(30):
    for X_batch, y_batch in target_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_ft.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer_ft.step()

# Evaluate on target AFTER fine-tuning
model.eval()
with torch.no_grad():
    preds_after = model(X_target.to(device)).argmax(dim=1)
    acc_after = (preds_after == y_target.to(device)).float().mean().item()
print(f'Target accuracy AFTER fine-tuning: {acc_after:.3f}')
print(f'\nImprovement: {acc_after - acc_before:.3f}')

### Discussion: Fine-tuning

Key observations:
- The pretrained model already has some useful features from the source domain
- Fine-tuning with a **lower learning rate** preserves useful features while adapting
- With only 50 target samples, training from scratch would likely overfit severely
- The improvement demonstrates positive transfer from source to target

---
## 3. Conservation Learning

### Concept

Conservation learning addresses **catastrophic forgetting** by constraining the new model
to stay close to the old (pretrained) model during fine-tuning.

### Two Approaches

**1. Output Closeness (Knowledge Distillation)**

Constrain the new model's outputs to be similar to the old model's outputs:

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{task}}(y, \hat{y}) + \lambda \cdot \text{KL}(f_{\text{old}}(x) \| f_{\text{new}}(x))$$

**2. Parameter Closeness (L2 Regularization)**

Constrain the new model's parameters to stay close to the old model's parameters:

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{task}} + \lambda \sum_i \|\theta_i^{\text{new}} - \theta_i^{\text{old}}\|_2^2$$

This penalizes large deviations from the pretrained weights, preventing the model from
drifting too far from its source knowledge.

In [ ]:
# Conservation Learning: Parameter Closeness (L2 constraint)

class ConservativeFinetuner:
    """Fine-tune while keeping parameters close to pretrained model."""
    
    def __init__(self, model, pretrained_params, lambda_reg=0.1):
        self.model = model
        self.pretrained_params = pretrained_params  # Saved pretrained weights
        self.lambda_reg = lambda_reg
    
    def conservation_loss(self):
        """L2 penalty: ||theta_new - theta_old||^2"""
        loss = 0.0
        for (name, param), old_param in zip(
            self.model.named_parameters(), self.pretrained_params
        ):
            loss += torch.sum((param - old_param) ** 2)
        return self.lambda_reg * loss

# Create and pretrain a model
torch.manual_seed(42)
model_cons = SimpleNet().to(device)
optimizer = optim.Adam(model_cons.parameters(), lr=0.01)

# Pretrain on source
for epoch in range(20):
    for X_batch, y_batch in source_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model_cons(X_batch), y_batch)
        loss.backward()
        optimizer.step()

# Save pretrained parameters
pretrained_params = [p.clone().detach() for p in model_cons.parameters()]

# Fine-tune WITH conservation constraint
finetuner = ConservativeFinetuner(model_cons, pretrained_params, lambda_reg=0.5)
optimizer_cons = optim.Adam(model_cons.parameters(), lr=0.001)

print('Fine-tuning with conservation constraint (lambda=0.5)...')
losses_task = []
losses_reg = []

model_cons.train()
for epoch in range(50):
    for X_batch, y_batch in target_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_cons.zero_grad()
        
        # Task loss
        task_loss = criterion(model_cons(X_batch), y_batch)
        # Conservation loss
        reg_loss = finetuner.conservation_loss()
        # Total loss
        total_loss = task_loss + reg_loss
        
        total_loss.backward()
        optimizer_cons.step()
    
    losses_task.append(task_loss.item())
    losses_reg.append(reg_loss.item())

# Measure parameter drift
param_drift = sum(
    torch.sum((p - p_old) ** 2).item()
    for p, p_old in zip(model_cons.parameters(), pretrained_params)
)
print(f'Total parameter drift (L2): {param_drift:.4f}')

# Plot losses
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(losses_task, label='Task Loss')
ax1.plot(losses_reg, label='Conservation Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Conservation Learning: Loss Components')
ax1.legend()

# Compare parameter distributions
old_params = torch.cat([p.flatten() for p in pretrained_params]).cpu().numpy()
new_params = torch.cat([p.flatten().detach() for p in model_cons.parameters()]).cpu().numpy()
ax2.hist(old_params, bins=50, alpha=0.5, label='Pretrained', density=True)
ax2.hist(new_params, bins=50, alpha=0.5, label='After Fine-tuning', density=True)
ax2.set_xlabel('Parameter Value')
ax2.set_ylabel('Density')
ax2.set_title('Parameter Distribution: Old vs New')
ax2.legend()

plt.tight_layout()
plt.show()
print('Conservation learning keeps parameters close to pretrained values.')

### Discussion: Conservation Learning

- The **conservation loss** acts as a regularizer that prevents catastrophic forgetting
- Higher $\lambda$ → parameters stay closer to pretrained values (more conservative)
- Lower $\lambda$ → more freedom to adapt (risk of forgetting)
- The parameter distribution plot shows that fine-tuned weights remain close to pretrained ones
- This is especially useful when the target domain is very different from the source

---
## 4. Layer Transfer

### Concept

Instead of fine-tuning the entire model, we can **copy specific layers** from the pretrained
model and only train the remaining layers.

### Which Layers to Transfer?

This depends on the domain:

| Domain | Layers to Copy | Reasoning |
|--------|---------------|------------|
| **Computer Vision** | First layers (early) | Early layers learn universal features (edges, textures) |
| **Speech Recognition** | Last layers (later) | Later layers capture speaker-independent phonetic features |
| **NLP** | First layers (early) | Early layers learn general language structure |

### Process
1. Train model on source task
2. Copy selected layers to new model (freeze them)
3. Randomly initialize remaining layers
4. Train only the new layers on target data

### Why Different Layers for Different Domains?
- **Vision:** Early layers detect edges/textures (universal), later layers detect task-specific objects
- **Speech:** Early layers capture speaker-specific acoustics, later layers capture language structure

In [ ]:
# Layer Transfer: Freeze early layers, train later layers

class DeepNet(nn.Module):
    """A deeper network to demonstrate layer transfer."""
    def __init__(self, input_dim=10, hidden_dim=64, output_dim=2):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)   # Early layer
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)  # Middle layer
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)  # Later layer
        self.layer4 = nn.Linear(hidden_dim, output_dim)  # Output layer
    
    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        x = F.relu(self.layer3(x))
        return self.layer4(x)

# Pretrain on source
torch.manual_seed(42)
source_model = DeepNet().to(device)
optimizer = optim.Adam(source_model.parameters(), lr=0.01)

for epoch in range(30):
    for X_batch, y_batch in source_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(source_model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

print('Source model trained.')

# Create target model and copy early layers (vision-style transfer)
target_model = DeepNet().to(device)

# Copy layers 1 and 2 from source (freeze them)
target_model.layer1.load_state_dict(source_model.layer1.state_dict())
target_model.layer2.load_state_dict(source_model.layer2.state_dict())

# Freeze copied layers
for param in target_model.layer1.parameters():
    param.requires_grad = False
for param in target_model.layer2.parameters():
    param.requires_grad = False

# Only train layers 3 and 4
trainable_params = [p for p in target_model.parameters() if p.requires_grad]
optimizer_target = optim.Adam(trainable_params, lr=0.01)

print(f'\nTotal parameters: {sum(p.numel() for p in target_model.parameters())}')
print(f'Trainable parameters: {sum(p.numel() for p in trainable_params)}')
print(f'Frozen parameters: {sum(p.numel() for p in target_model.parameters() if not p.requires_grad)}')

# Train on target
print('\nTraining only later layers on target data...')
target_model.train()
for epoch in range(30):
    for X_batch, y_batch in target_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_target.zero_grad()
        loss = criterion(target_model(X_batch), y_batch)
        loss.backward()
        optimizer_target.step()

# Evaluate
target_model.eval()
with torch.no_grad():
    preds = target_model(X_target.to(device)).argmax(dim=1)
    acc = (preds == y_target.to(device)).float().mean().item()
print(f'\nTarget accuracy with layer transfer: {acc:.3f}')

In [ ]:
# Visualize which layers are frozen vs trainable
fig, ax = plt.subplots(figsize=(10, 3))

layers = ['Layer 1\n(Input→Hidden)', 'Layer 2\n(Hidden→Hidden)', 
          'Layer 3\n(Hidden→Hidden)', 'Layer 4\n(Hidden→Output)']
colors = ['#3498db', '#3498db', '#e74c3c', '#e74c3c']  # Blue=frozen, Red=trainable
labels = ['Copied & Frozen\n(from source)', 'Copied & Frozen\n(from source)',
           'Randomly Init\n(trainable)', 'Randomly Init\n(trainable)']

bars = ax.barh(range(4), [1]*4, color=colors, edgecolor='black', height=0.6)
for i, (layer, label) in enumerate(zip(layers, labels)):
    ax.text(0.5, i, f'{layer}\n{label}', ha='center', va='center', fontsize=9, fontweight='bold')

ax.set_xlim(0, 1)
ax.set_yticks([])
ax.set_xticks([])
ax.set_title('Layer Transfer (Vision-style): Early Layers Frozen, Later Layers Trained', fontsize=12)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#3498db', label='Frozen (transferred from source)'),
                   Patch(facecolor='#e74c3c', label='Trainable (trained on target)')]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

---
## 5. Multitask Learning

### Concept

Multitask learning trains a single model on **multiple related tasks simultaneously**.
The model shares hidden layers across tasks but has task-specific output heads.

### Architecture
```
Input → [Shared Hidden Layers] → Task 1 Output Head → Task 1 Prediction
                                → Task 2 Output Head → Task 2 Prediction
                                → Task 3 Output Head → Task 3 Prediction
```

### Benefits
- **Implicit data augmentation:** Each task provides additional training signal
- **Regularization:** Learning multiple tasks prevents overfitting to any single task
- **Feature sharing:** Tasks can benefit from features learned by other tasks

### Example: Multilingual Speech Recognition
- Shared layers learn universal phonetic features
- Language-specific output heads handle different phoneme sets
- Low-resource languages benefit from high-resource language data

In [ ]:
# Multitask Learning: Shared backbone with task-specific heads

class MultitaskNet(nn.Module):
    """Shared hidden layers with multiple task-specific output heads."""
    
    def __init__(self, input_dim=10, hidden_dim=64, num_tasks=3, classes_per_task=2):
        super().__init__()
        # Shared layers
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        # Task-specific heads
        self.heads = nn.ModuleList([
            nn.Linear(hidden_dim, classes_per_task) for _ in range(num_tasks)
        ])
    
    def forward(self, x, task_id):
        """Forward pass for a specific task."""
        shared_features = self.shared(x)
        return self.heads[task_id](shared_features)
    
    def forward_all(self, x):
        """Forward pass for all tasks (returns list of outputs)."""
        shared_features = self.shared(x)
        return [head(shared_features) for head in self.heads]

# Create multitask data (3 related binary classification tasks)
torch.manual_seed(42)
X_multi = torch.randn(500, 10)

# Tasks share some structure but have different decision boundaries
y_task0 = (X_multi[:, 0] + X_multi[:, 1] > 0).long()       # Uses features 0, 1
y_task1 = (X_multi[:, 1] + X_multi[:, 2] > 0).long()       # Uses features 1, 2
y_task2 = (X_multi[:, 0] + X_multi[:, 2] > 0).long()       # Uses features 0, 2
labels = [y_task0, y_task1, y_task2]

# Train multitask model
mtl_model = MultitaskNet(num_tasks=3).to(device)
optimizer = optim.Adam(mtl_model.parameters(), lr=0.01)

print('Training multitask model on 3 tasks simultaneously...')
task_losses_history = {0: [], 1: [], 2: []}

for epoch in range(50):
    mtl_model.train()
    epoch_losses = {0: 0, 1: 0, 2: 0}
    
    # Sample batches for each task
    indices = torch.randperm(500)[:64]
    X_batch = X_multi[indices].to(device)
    
    optimizer.zero_grad()
    total_loss = 0
    
    for task_id in range(3):
        y_batch = labels[task_id][indices].to(device)
        output = mtl_model(X_batch, task_id)
        task_loss = criterion(output, y_batch)
        total_loss += task_loss
        epoch_losses[task_id] = task_loss.item()
    
    total_loss.backward()
    optimizer.step()
    
    for t in range(3):
        task_losses_history[t].append(epoch_losses[t])

# Evaluate each task
mtl_model.eval()
print('\nTask Accuracies:')
with torch.no_grad():
    X_all = X_multi.to(device)
    for task_id in range(3):
        preds = mtl_model(X_all, task_id).argmax(dim=1)
        acc = (preds == labels[task_id].to(device)).float().mean().item()
        print(f'  Task {task_id}: {acc:.3f}')

# Plot training curves
plt.figure(figsize=(8, 4))
for t in range(3):
    plt.plot(task_losses_history[t], label=f'Task {t}')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Multitask Learning: Per-Task Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('All tasks benefit from shared feature learning.')

---
## 6. Domain-Adversarial Training

### Concept

Domain-adversarial training learns **domain-invariant features** by using an adversarial
objective. The model has three components:

1. **Feature Extractor** $G_f$: Maps input to feature representation
2. **Label Predictor** $G_y$: Predicts task labels from features
3. **Domain Classifier** $G_d$: Predicts which domain the input came from

### Adversarial Objective

$$\min_{G_f, G_y} \max_{G_d} \; \mathcal{L}_y(G_y(G_f(x)), y) - \lambda \cdot \mathcal{L}_d(G_d(G_f(x)), d)$$

- **Maximize** label prediction accuracy (minimize $\mathcal{L}_y$)
- **Minimize** domain classification accuracy (maximize $\mathcal{L}_d$ for the feature extractor)

The feature extractor learns representations that are:
- **Useful** for the task (good label prediction)
- **Domain-invariant** (domain classifier cannot distinguish source from target)

### Gradient Reversal Layer (GRL)
During backpropagation, the gradient from the domain classifier is **reversed** before
reaching the feature extractor. This makes the feature extractor adversarial to the domain classifier.

In [ ]:
# Domain-Adversarial Training with Gradient Reversal Layer

class GradientReversalFunction(torch.autograd.Function):
    """Gradient Reversal Layer - reverses gradient during backprop."""
    @staticmethod
    def forward(ctx, x, lambda_val):
        ctx.lambda_val = lambda_val
        return x.clone()
    
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_val * grad_output, None

class GRL(nn.Module):
    def __init__(self, lambda_val=1.0):
        super().__init__()
        self.lambda_val = lambda_val
    
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_val)

class DomainAdversarialNet(nn.Module):
    """Domain-adversarial neural network (DANN)."""
    
    def __init__(self, input_dim=10, feature_dim=32, num_classes=2, lambda_val=1.0):
        super().__init__()
        # Feature extractor
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, feature_dim),
            nn.ReLU(),
        )
        # Label predictor
        self.label_predictor = nn.Sequential(
            nn.Linear(feature_dim, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes),
        )
        # Domain classifier (with GRL)
        self.grl = GRL(lambda_val)
        self.domain_classifier = nn.Sequential(
            nn.Linear(feature_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 2),  # 2 domains: source vs target
        )
    
    def forward(self, x):
        features = self.feature_extractor(x)
        label_output = self.label_predictor(features)
        # GRL reverses gradients for domain classifier
        domain_output = self.domain_classifier(self.grl(features))
        return label_output, domain_output, features

# Create source and target data with domain shift
torch.manual_seed(42)
n_source, n_target = 300, 100

# Source domain
X_src = torch.randn(n_source, 10)
y_src = (X_src[:, 0] + X_src[:, 1] > 0).long()
d_src = torch.zeros(n_source).long()  # domain label = 0

# Target domain (shifted distribution, same underlying task)
X_tgt = torch.randn(n_target, 10) * 1.5 + 0.5  # Different distribution!
y_tgt = (X_tgt[:, 0] + X_tgt[:, 1] > 0.5).long()  # Similar task
d_tgt = torch.ones(n_target).long()  # domain label = 1

# Train DANN
dann = DomainAdversarialNet(lambda_val=1.0).to(device)
optimizer = optim.Adam(dann.parameters(), lr=0.005)

print('Training Domain-Adversarial Network...')
label_losses = []
domain_losses = []

for epoch in range(100):
    dann.train()
    
    # Sample batches
    src_idx = torch.randperm(n_source)[:32]
    tgt_idx = torch.randperm(n_target)[:32]
    
    X_batch = torch.cat([X_src[src_idx], X_tgt[tgt_idx]]).to(device)
    d_batch = torch.cat([d_src[src_idx], d_tgt[tgt_idx]]).to(device)
    
    optimizer.zero_grad()
    label_out, domain_out, features = dann(X_batch)
    
    # Label loss (only on source data where we have labels)
    label_loss = criterion(label_out[:32], y_src[src_idx].to(device))
    # Domain loss (on all data)
    domain_loss = criterion(domain_out, d_batch)
    
    # Total loss (GRL handles the adversarial part automatically)
    total_loss = label_loss + domain_loss
    total_loss.backward()
    optimizer.step()
    
    label_losses.append(label_loss.item())
    domain_losses.append(domain_loss.item())

# Evaluate
dann.eval()
with torch.no_grad():
    label_out, domain_out, features = dann(X_tgt.to(device))
    target_acc = (label_out.argmax(1) == y_tgt.to(device)).float().mean().item()
    domain_acc = (domain_out.argmax(1) == d_tgt.to(device)).float().mean().item()

print(f'\nTarget label accuracy: {target_acc:.3f}')
print(f'Domain classifier accuracy: {domain_acc:.3f} (closer to 0.5 = more domain-invariant)')

In [ ]:
# Visualize domain-adversarial training results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss curves
axes[0].plot(label_losses, label='Label Loss', color='blue')
axes[0].plot(domain_losses, label='Domain Loss', color='red')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('DANN Training Losses')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Feature space visualization (2D projection)
dann.eval()
with torch.no_grad():
    _, _, feat_src = dann(X_src.to(device))
    _, _, feat_tgt = dann(X_tgt.to(device))
    feat_src = feat_src.cpu().numpy()
    feat_tgt = feat_tgt.cpu().numpy()

# Use first 2 feature dimensions for visualization
axes[1].scatter(feat_src[:, 0], feat_src[:, 1], c='blue', alpha=0.4, label='Source', s=20)
axes[1].scatter(feat_tgt[:, 0], feat_tgt[:, 1], c='red', alpha=0.4, label='Target', s=20)
axes[1].set_xlabel('Feature Dim 1')
axes[1].set_ylabel('Feature Dim 2')
axes[1].set_title('Learned Features (Domain-Invariant)')
axes[1].legend()

# Architecture diagram
axes[2].axis('off')
axes[2].set_title('DANN Architecture', fontsize=12)
boxes = [
    (0.5, 0.85, 'Input x', 'lightyellow'),
    (0.5, 0.65, 'Feature Extractor $G_f$', 'lightblue'),
    (0.25, 0.4, 'Label Predictor $G_y$', 'lightgreen'),
    (0.75, 0.4, 'Domain Classifier $G_d$', 'lightsalmon'),
    (0.25, 0.2, 'Task Label $\hat{y}$', 'white'),
    (0.75, 0.2, 'Domain Label $\hat{d}$', 'white'),
]
for x, y, text, color in boxes:
    axes[2].text(x, y, text, ha='center', va='center', fontsize=9,
                bbox=dict(boxstyle='round', facecolor=color, edgecolor='black'))

# Arrows
axes[2].annotate('', xy=(0.5, 0.72), xytext=(0.5, 0.82), arrowprops=dict(arrowstyle='->', lw=1.5))
axes[2].annotate('', xy=(0.25, 0.47), xytext=(0.4, 0.6), arrowprops=dict(arrowstyle='->', lw=1.5))
axes[2].annotate('', xy=(0.75, 0.47), xytext=(0.6, 0.6), arrowprops=dict(arrowstyle='->', lw=1.5, color='red'))
axes[2].annotate('', xy=(0.25, 0.27), xytext=(0.25, 0.35), arrowprops=dict(arrowstyle='->', lw=1.5))
axes[2].annotate('', xy=(0.75, 0.27), xytext=(0.75, 0.35), arrowprops=dict(arrowstyle='->', lw=1.5))
axes[2].text(0.75, 0.55, 'GRL\n(reverse\ngradient)', ha='center', fontsize=7, color='red')

plt.tight_layout()
plt.show()
print('The GRL makes features domain-invariant by confusing the domain classifier.')

---
## 7. Zero-shot and Few-shot Learning

### Zero-shot Learning

Classify examples from classes **never seen during training**. Instead of learning
class-specific features, the model learns a **similarity function** or maps inputs
to a semantic embedding space.

### Few-shot Learning

Learn to classify with only **a few examples per class** (typically 1-5).

### k-way n-shot Setup
- **k-way:** k classes to distinguish between
- **n-shot:** n examples per class in the support set
- **Support set:** The few labelled examples available
- **Query set:** New examples to classify

### Approach: Learning Similarity

Instead of learning $P(y|x)$ directly, learn a similarity function $\text{sim}(x_1, x_2)$:
1. Embed both query and support examples into a feature space
2. Compare query embedding to each support class embedding
3. Assign the class of the most similar support example

### Prototypical Networks
- Compute class **prototypes** as the mean embedding of support examples
- Classify query by finding the nearest prototype

$$c_k = \frac{1}{|S_k|} \sum_{x_i \in S_k} f_\theta(x_i)$$
$$P(y=k|x) = \frac{\exp(-d(f_\theta(x), c_k))}{\sum_{k'} \exp(-d(f_\theta(x), c_{k'}))}$$

In [ ]:
# Few-shot Learning with Prototypical Networks

class EmbeddingNet(nn.Module):
    """Embedding network for few-shot learning."""
    def __init__(self, input_dim=10, embed_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, embed_dim),
        )
    
    def forward(self, x):
        return self.encoder(x)

class PrototypicalNetwork:
    """Prototypical Network for few-shot classification."""
    
    def __init__(self, embedding_net):
        self.embedding_net = embedding_net
    
    def compute_prototypes(self, support_set, support_labels):
        """Compute class prototypes from support set."""
        embeddings = self.embedding_net(support_set)
        classes = torch.unique(support_labels)
        prototypes = []
        for c in classes:
            mask = support_labels == c
            prototype = embeddings[mask].mean(dim=0)
            prototypes.append(prototype)
        return torch.stack(prototypes), classes
    
    def classify(self, query, support_set, support_labels):
        """Classify query examples using nearest prototype."""
        prototypes, classes = self.compute_prototypes(support_set, support_labels)
        query_embeddings = self.embedding_net(query)
        
        # Compute distances to each prototype
        distances = torch.cdist(query_embeddings, prototypes)  # (n_query, n_classes)
        # Classify by nearest prototype
        pred_indices = distances.argmin(dim=1)
        return classes[pred_indices]

# Demonstrate few-shot learning
torch.manual_seed(42)

# Create 5 classes (5-way classification)
n_classes = 5
n_shot = 3      # 3 examples per class in support set
n_query = 10    # 10 query examples per class

# Generate class-specific data
class_centers = torch.randn(n_classes, 10) * 3  # Random class centers

support_data = []
support_labels_list = []
query_data = []
query_labels_list = []

for c in range(n_classes):
    # Support set: n_shot examples per class
    support = class_centers[c] + torch.randn(n_shot, 10) * 0.5
    support_data.append(support)
    support_labels_list.append(torch.full((n_shot,), c))
    
    # Query set: n_query examples per class
    query = class_centers[c] + torch.randn(n_query, 10) * 0.5
    query_data.append(query)
    query_labels_list.append(torch.full((n_query,), c))

support_set = torch.cat(support_data).to(device)
support_labels = torch.cat(support_labels_list).to(device)
query_set = torch.cat(query_data).to(device)
query_labels = torch.cat(query_labels_list).to(device)

print(f'Few-shot setup: {n_classes}-way {n_shot}-shot')
print(f'Support set: {support_set.shape[0]} examples ({n_shot} per class)')
print(f'Query set: {query_set.shape[0]} examples ({n_query} per class)')

# Train embedding network with episodic training
embed_net = EmbeddingNet(input_dim=10, embed_dim=32).to(device)
proto_net = PrototypicalNetwork(embed_net)
optimizer = optim.Adam(embed_net.parameters(), lr=0.01)

print('\nTraining prototypical network...')
for episode in range(200):
    embed_net.train()
    optimizer.zero_grad()
    
    # Compute prototypes
    prototypes, classes = proto_net.compute_prototypes(support_set, support_labels)
    
    # Embed queries
    query_embeddings = embed_net(query_set)
    
    # Compute negative log-probability
    distances = torch.cdist(query_embeddings, prototypes)  # (n_query_total, n_classes)
    log_probs = F.log_softmax(-distances, dim=1)
    loss = F.nll_loss(log_probs, query_labels)
    
    loss.backward()
    optimizer.step()

# Evaluate
embed_net.eval()
with torch.no_grad():
    predictions = proto_net.classify(query_set, support_set, support_labels)
    accuracy = (predictions == query_labels).float().mean().item()

print(f'\nFew-shot accuracy ({n_classes}-way {n_shot}-shot): {accuracy:.3f}')
print(f'Random baseline: {1/n_classes:.3f}')

In [ ]:
# Visualize few-shot learning: embeddings and prototypes
from sklearn.decomposition import PCA

embed_net.eval()
with torch.no_grad():
    support_embeddings = embed_net(support_set).cpu().numpy()
    query_embeddings = embed_net(query_set).cpu().numpy()
    prototypes_np, _ = proto_net.compute_prototypes(support_set, support_labels)
    prototypes_np = prototypes_np.cpu().numpy()

# PCA to 2D for visualization
all_embeddings = np.vstack([support_embeddings, query_embeddings, prototypes_np])
pca = PCA(n_components=2)
all_2d = pca.fit_transform(all_embeddings)

n_support = support_embeddings.shape[0]
n_query_total = query_embeddings.shape[0]
support_2d = all_2d[:n_support]
query_2d = all_2d[n_support:n_support + n_query_total]
proto_2d = all_2d[n_support + n_query_total:]

fig, ax = plt.subplots(figsize=(8, 6))
colors = plt.cm.Set1(np.linspace(0, 1, n_classes))

for c in range(n_classes):
    # Support examples (squares)
    s_mask = support_labels.cpu().numpy() == c
    ax.scatter(support_2d[s_mask, 0], support_2d[s_mask, 1], 
              c=[colors[c]], marker='s', s=100, edgecolors='black', 
              label=f'Class {c} (support)' if c == 0 else None, zorder=3)
    
    # Query examples (circles)
    q_mask = query_labels.cpu().numpy() == c
    ax.scatter(query_2d[q_mask, 0], query_2d[q_mask, 1],
              c=[colors[c]], marker='o', s=30, alpha=0.5, zorder=2)
    
    # Prototype (star)
    ax.scatter(proto_2d[c, 0], proto_2d[c, 1],
              c=[colors[c]], marker='*', s=300, edgecolors='black', linewidths=1.5, zorder=4)

ax.set_xlabel('PCA Dimension 1')
ax.set_ylabel('PCA Dimension 2')
ax.set_title(f'Prototypical Network: {n_classes}-way {n_shot}-shot\n'
             f'(★=prototypes, ■=support, ●=query)', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Prototypes (stars) are the mean of support embeddings (squares).')
print('Queries (circles) are classified by nearest prototype.')

---
## 8. Practical Example: Fine-tuning a Pretrained Model

### Task: Transfer Learning with a Pretrained CNN

We will fine-tune a pretrained ResNet-18 (trained on ImageNet) to classify a small
custom dataset. This demonstrates the full transfer learning pipeline:

1. Load pretrained model
2. Replace the final classification layer
3. Freeze early layers (optional)
4. Fine-tune on target data

We use CIFAR-10 with only 100 samples per class to simulate a limited-data scenario.

In [ ]:
# Practical Transfer Learning: Fine-tune ResNet-18 on small CIFAR-10 subset

# Define transforms (ResNet expects 224x224, but we'll use 32x32 with adaptation)
transform_train = transforms.Compose([
    transforms.Resize(32),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

# Download CIFAR-10
print('Downloading CIFAR-10...')
full_trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train
)
testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test
)

# Create a SMALL subset (100 samples per class = 1000 total)
# This simulates limited target data
samples_per_class = 100
class_counts = {i: 0 for i in range(10)}
subset_indices = []

for idx in range(len(full_trainset)):
    _, label = full_trainset[idx]
    if class_counts[label] < samples_per_class:
        subset_indices.append(idx)
        class_counts[label] += 1
    if all(c >= samples_per_class for c in class_counts.values()):
        break

small_trainset = torch.utils.data.Subset(full_trainset, subset_indices)
train_loader = DataLoader(small_trainset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

print(f'Training samples: {len(small_trainset)} (only {samples_per_class} per class)')
print(f'Test samples: {len(testset)}')
print(f'Classes: {full_trainset.classes}')

In [ ]:
# Compare: Training from scratch vs Fine-tuning pretrained model

def create_model(pretrained=True):
    """Create ResNet-18 for CIFAR-10 (10 classes)."""
    if pretrained:
        # Load pretrained weights (trained on ImageNet)
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    else:
        model = models.resnet18(weights=None)
    
    # Modify first conv layer for 32x32 input (CIFAR-10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()  # Remove maxpool for small images
    
    # Replace final FC layer for 10 classes
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 10)
    
    return model.to(device)

def train_model(model, train_loader, epochs=10, lr=0.001):
    """Train model and return accuracy history."""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    criterion = nn.CrossEntropyLoss()
    
    train_accs = []
    
    for epoch in range(epochs):
        model.train()
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        scheduler.step()
        train_acc = correct / total
        train_accs.append(train_acc)
        
        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1}/{epochs} - Train Acc: {train_acc:.3f}')
    
    return train_accs

def evaluate_model(model, test_loader):
    """Evaluate model on test set."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return correct / total

# Train from scratch
print('=== Training from SCRATCH (no pretraining) ===')
model_scratch = create_model(pretrained=False)
accs_scratch = train_model(model_scratch, train_loader, epochs=15, lr=0.001)
test_acc_scratch = evaluate_model(model_scratch, test_loader)
print(f'  Test Accuracy: {test_acc_scratch:.3f}\n')

# Fine-tune pretrained model
print('=== FINE-TUNING pretrained ResNet-18 ===')
model_pretrained = create_model(pretrained=True)
accs_pretrained = train_model(model_pretrained, train_loader, epochs=15, lr=0.001)
test_acc_pretrained = evaluate_model(model_pretrained, test_loader)
print(f'  Test Accuracy: {test_acc_pretrained:.3f}\n')

print(f'\n=== RESULTS ===')
print(f'From scratch:  {test_acc_scratch:.3f}')
print(f'Fine-tuned:    {test_acc_pretrained:.3f}')
print(f'Improvement:   +{test_acc_pretrained - test_acc_scratch:.3f}')

In [ ]:
# Visualize the comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Training curves
epochs_range = range(1, len(accs_scratch) + 1)
ax1.plot(epochs_range, accs_scratch, 'b-o', label='From Scratch', markersize=4)
ax1.plot(epochs_range, accs_pretrained, 'r-o', label='Fine-tuned (Pretrained)', markersize=4)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Accuracy')
ax1.set_title('Training Progress: Scratch vs Fine-tuning')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1)

# Test accuracy comparison
methods = ['From Scratch', 'Fine-tuned\n(Pretrained)']
accuracies = [test_acc_scratch, test_acc_pretrained]
bars = ax2.bar(methods, accuracies, color=['steelblue', 'coral'], edgecolor='black')
ax2.set_ylabel('Test Accuracy')
ax2.set_title('Test Accuracy Comparison\n(1000 training samples)')
ax2.set_ylim(0, 1)
for bar, acc in zip(bars, accuracies):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{acc:.3f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()
print('\nKey takeaway: Pretrained models converge faster and generalize better with limited data.')

### Discussion: Practical Fine-tuning

Key observations from this experiment:

1. **Faster convergence:** The pretrained model reaches high accuracy much sooner
2. **Better generalization:** With only 1000 training samples, the pretrained model
   significantly outperforms training from scratch on the test set
3. **Less overfitting:** The pretrained features act as a strong regularizer

### Best Practices for Fine-tuning
- Use a **lower learning rate** than training from scratch (pretrained features are already good)
- Consider **freezing early layers** if target data is very limited
- Use **learning rate warmup** to avoid destroying pretrained features early on
- **Data augmentation** helps prevent overfitting on small target datasets

---
## 9. Summary

### Transfer Learning Methods Covered

| Method | Key Idea | When to Use |
|--------|----------|-------------|
| **Fine-tuning** | Train on source, then adapt to target | Most common; good default approach |
| **Conservation Learning** | Constrain parameters to stay close to pretrained | When catastrophic forgetting is a concern |
| **Layer Transfer** | Copy specific layers, train the rest | When you know which features transfer |
| **Multitask Learning** | Share layers across tasks | When multiple related tasks are available |
| **Domain-Adversarial** | Learn domain-invariant features | When source and target distributions differ |
| **Few-shot Learning** | Learn similarity, not classes | When very few target examples exist |

### Key Takeaways

1. Transfer learning is essential when target data is limited
2. The choice of method depends on:
   - How similar source and target domains are
   - How much target data is available
   - Whether source and target tasks are the same
3. Pretrained models (ImageNet, language models) provide excellent starting points
4. Fine-tuning with a lower learning rate is the simplest and often most effective approach
5. Domain-adversarial training is powerful when there is a clear domain shift

### Connection to COMPSCI 714

These techniques are fundamental to modern AI systems:
- Large language models (GPT, BERT) are pretrained then fine-tuned
- Speech recognition systems use multilingual pretraining
- Computer vision models transfer ImageNet features to specialized tasks
- Few-shot learning enables rapid adaptation to new tasks

---
*Notebook aligned with COMPSCI 714 Lecture 7-2: Transfer Learning II*